# 🤖 Autonomous B2B Sales Pipeline with AI Agents
### Built with CrewAI, Claude (Anthropic), Tavily, and Gmail API

This notebook demonstrates a fully autonomous B2B outbound sales pipeline powered by AI agents. 

Given a target market, the pipeline:
1. **Researches** companies showing signs of pain relevant to your product
2. **Finds** the right decision-maker contacts at each company
3. **Writes** personalized cold outreach emails for each contact
4. **Creates** Gmail drafts ready to review and send

**Stack:** Python · CrewAI · Claude Sonnet (Anthropic) · Tavily Search · Gmail API

---


## 1. Setup & Imports

We import all required libraries and configure our environment.

- `crewai` — agent orchestration framework
- `TavilySearchTool` — real-time web search for agents
- `google-auth` libraries — Gmail API authentication
- `dotenv` — loads API keys from `.env` file securely


In [2]:
import os
import os.path
import base64
import re
from email.mime.text import MIMEText
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM
from crewai_tools import TavilySearchTool

print("✅ All libraries imported successfully")


✅ All libraries imported successfully


## 2. Gmail Authentication

We set up OAuth2 authentication with Gmail using Google's API.

- **`SCOPES`** — defines what permissions we request (compose only)
- **`get_gmail_service()`** — handles the full OAuth flow, caches the token locally
- **`create_draft()`** — formats and pushes an email to Gmail drafts

The first time this runs, it opens a browser window to authorize access. After that, it uses a cached `token.json` automatically.


In [3]:
# Gmail OAuth scope — compose only (least privilege)
SCOPES = ['https://www.googleapis.com/auth/gmail.compose']

def get_gmail_service():
    """Authenticate with Gmail API and return a service object."""
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

def create_draft(service, to, subject, body):
    """Create a Gmail draft for a given recipient, subject, and body."""
    message = MIMEText(body)
    message['to'] = to
    message['subject'] = subject
    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()
    draft = service.users().drafts().create(
        userId='me', 
        body={'message': {'raw': raw}}
    ).execute()
    print(f"✅ Draft created for {to}: {subject}")
    return draft

print("✅ Gmail functions defined")


✅ Gmail functions defined


## 3. Load Environment & Initialize LLM

We load API keys from a `.env` file and initialize Claude as the LLM for all agents.

**Why Claude?**
- Strong reasoning for multi-step research tasks
- Excellent at nuanced, personalized copywriting
- Native tool use support via Anthropic API

**Why the OpenAI placeholder?**
CrewAI defaults to OpenAI and requires the key to exist even when using a different provider. We set a placeholder to satisfy this requirement without using OpenAI at all.


In [4]:
# Load API keys from .env file
load_dotenv()

# Satisfy CrewAI's OpenAI requirement without using it
os.environ["OPENAI_API_KEY"] = "sk-placeholder"

# Initialize Claude Sonnet as the LLM for all agents
claude = LLM(
    model="anthropic/claude-sonnet-4-6",
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

# Initialize Tavily for real-time web search
search_tool = TavilySearchTool()

print("✅ Claude LLM initialized")
print("✅ Tavily search tool ready")


✅ Claude LLM initialized
✅ Tavily search tool ready


## 4. Agent 1 — Tech Industry Researcher

The first agent scans the web for mid-size tech companies showing public signals of project management pain.

**What it searches for:**
- News about missed deadlines or failed product launches
- Glassdoor reviews mentioning coordination issues
- Job postings signaling rapid team growth
- Press coverage of restructuring or layoffs

**Tools:** Tavily web search  
**Output:** List of 5+ target companies with pain point summaries


In [5]:
researcher = Agent(
    role="Tech Industry Researcher",
    goal="Find mid-size tech companies struggling with project management, missed deadlines, failed product launches, or rapid team growth",
    backstory="""You are an expert tech industry researcher. 
    You know how to find companies that are publicly signaling project 
    management pain through news, job postings, Glassdoor reviews, 
    LinkedIn posts, and press releases.""",
    tools=[search_tool],
    llm=claude,
    verbose=True
)

research_task = Task(
    description="""Search for mid-size tech companies (50-1000 employees) in the US 
    that are dealing with project management challenges such as missed deadlines, 
    failed product launches, rapid team growth, or remote team coordination issues.
    Look for recent news, job postings, Glassdoor reviews, and press coverage.
    Return a list of at least 5 companies with a brief note on why each is a good lead.""",
    expected_output="A list of 5 tech companies with company name, location, size, and reason they are a good lead",
    agent=researcher
)

print("✅ Agent 1 (Researcher) defined")


✅ Agent 1 (Researcher) defined


## 5. Agent 2 — Lead Finder

The second agent takes the companies identified by Agent 1 and finds the right decision-maker contacts at each one.

**Contacts it looks for:**
- CEO / Co-Founder
- CTO / Chief Technology Officer
- VP of Engineering
- Head of Product / CPO
- Director of Operations

**Tools:** Tavily web search  
**Input:** Companies from Agent 1  
**Output:** Named contacts with titles and emails/LinkedIn URLs


In [6]:
lead_finder = Agent(
    role="Tech Company Lead Finder",
    goal="Find the right decision-maker contacts at each tech company identified by the researcher",
    backstory="""You are an expert at finding the right people to contact 
    at tech companies. You know how to identify CTOs, VPs of Engineering, 
    Heads of Product, Directors of Operations, and CEOs at mid-size tech 
    companies using public information and web searches.""",
    tools=[search_tool],
    llm=claude,
    verbose=True
)

lead_task = Task(
    description="""Using ONLY the specific companies identified by the researcher in the previous task,
    do not search for new companies. For each company already identified, find:
    - Full name and title of the CEO or Chief Executive Officer
    - Full name and title of the CTO or Chief Technology Officer
    - Full name and title of the VP of Engineering
    - Full name and title of the Head of Product or CPO
    - Full name and title of the Director of Operations
    - LinkedIn URL or professional email if publicly available""",
    expected_output="A list of contacts for each company from the researcher's list",
    agent=lead_finder,
    context=[research_task]
)

print("✅ Agent 2 (Lead Finder) defined")


✅ Agent 2 (Lead Finder) defined


## 6. Agent 3 — B2B Copywriter

The third agent writes personalized cold outreach emails for each contact identified by Agent 2.

**Each email:**
- References a specific, real pain point for that company
- Introduces ProjectFlow naturally without being salesy
- Has a compelling subject line
- Ends with a soft CTA (15-minute call)
- Stays under 150 words

**Tools:** None (pure reasoning and writing)  
**Input:** Companies from Agent 1 + Contacts from Agent 2  
**Output:** Personalized emails per contact per company


In [7]:
copywriter = Agent(
    role="B2B SaaS Copywriter",
    goal="Write personalized outreach emails to tech company decision-makers that speak directly to their project management pain",
    backstory="""You are an expert B2B SaaS copywriter. 
    You know how to write concise, personalized cold emails that resonate with 
    tech executives. You reference specific pain points unique to each company 
    and position ProjectFlow as directly relevant to their situation. 
    You never sound generic or salesy.""",
    tools=[],
    llm=claude,
    verbose=True
)

copy_task = Task(
    description="""Using ONLY the companies from the researcher and ONLY the contacts 
    from the lead finder, write personalized cold outreach emails.
    
    Do not invent new companies or contacts. Use exactly what was provided by the 
    previous two agents.
    
    For each company and its contacts write an email that:
    - Has a compelling subject line
    - Opens by referencing a specific pain point or news item about THAT company
    - Briefly introduces ProjectFlow: a project management platform that gives 
      engineering and product teams real-time visibility into sprint progress, 
      resource allocation, and delivery timelines
    - Includes a soft call to action (15 minute call)
    - Is under 150 words
    - Sounds human, not robotic or salesy""",
    expected_output="Personalized emails for each company using only the companies and contacts identified by the previous agents, organized by company name and contact",
    agent=copywriter,
    context=[research_task, lead_task]
)

print("✅ Agent 3 (Copywriter) defined")


✅ Agent 3 (Copywriter) defined


## 7. Agent 4 — Gmail Draft Creator

The fourth agent takes the emails written by Agent 3 and structures them for Gmail draft creation.

**What it does:**
- Extracts recipient, subject, and body from each email
- Formats everything cleanly for the Gmail API
- Returns a structured list ready for draft creation

**Tools:** None  
**Input:** Emails from Agent 3  
**Output:** Structured email list labeled by recipient and subject


In [8]:
gmail_drafter = Agent(
    role="Gmail Draft Creator",
    goal="Take the personalized emails written by the copywriter and create Gmail drafts for each one",
    backstory="""You are an expert at organizing and preparing email outreach. 
    You take finalized email copy and prepare it for sending by creating 
    Gmail drafts. You are precise and ensure every email is correctly 
    formatted with the right recipient, subject line, and body.""",
    tools=[],
    llm=claude,
    verbose=True
)

gmail_task = Task(
    description="""Take the emails written by the copywriter and create Gmail drafts.
    For each email extract:
    - The recipient contact name and email address
    - The subject line
    - The email body
    
    Then format them ready for Gmail draft creation.
    Return a structured list of all emails with recipient, subject, and body clearly separated.""",
    expected_output="A structured list of emails with recipient, subject line, and body clearly labeled for each contact",
    agent=gmail_drafter,
    context=[copy_task]
)

print("✅ Agent 4 (Gmail Drafter) defined")


✅ Agent 4 (Gmail Drafter) defined


## 8. Run the Pipeline

We assemble all 4 agents into a Crew and kick off the pipeline.

**Execution order:** Sequential — each agent waits for the previous one to finish before starting.

**What happens:**
1. Agent 1 searches the web for target companies
2. Agent 2 finds contacts at those specific companies
3. Agent 3 writes personalized emails for each contact
4. Agent 4 structures everything for Gmail

Each agent's output is saved to a `.txt` file so you never need to rerun expensive agents during development.


In [9]:
crew = Crew(
    agents=[researcher, lead_finder, copywriter, gmail_drafter],
    tasks=[research_task, lead_task, copy_task, gmail_task],
    verbose=True
)

# Run the pipeline
result = crew.kickoff()

# Save all outputs to files
for filename, task in [
    ("research_output.txt", research_task),
    ("leads_output.txt", lead_task),
    ("emails_output.txt", copy_task),
    ("gmail_output.txt", gmail_task)
]:
    try:
        with open(filename, "w") as f:
            f.write(task.output.raw)
        print(f"✅ {filename} saved")
    except Exception as e:
        print(f"❌ Could not save {filename}: {e}")

print("\n\n=== PIPELINE COMPLETE ===")
print(result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f43c7c1a-d8ad-440b-a133-c1f8ec1d15ed                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search for mid-size tech companies (50-1000 employees) in the US                                         │
│      that are dealing with project management challenges such as missed deadlines,                              │
│      failed product launches, rapid team growth, or remote team coordination issues.                            │
│      Look for recent news, job postings, Glassdoor reviews, and press coverage.                                 │
│      Return a list of at least 5 companies with a brief note on why each is a good lead.                        │
│  ID: 260c668e-c932-44d0-90f4-76d96c2316df                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Search for mid-size tech companies (50-1000 employees) in the US                                         │
│      that are dealing with project management challenges such as missed deadlines,                              │
│      failed product launches, rapid team growth, or remote team coordination issues.                            │
│      Look for recent news, job postings, Glassdoor reviews, and press coverage.                                 │
│      Return a list of at least 5 companies with a brief note on why each is a good lead.                        │
│  Agent: Tech Industry Researcher                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: f43c7c1a-d8ad-440b-a133-c1f8ec1d15ed                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.

## 9. Push Emails to Gmail Drafts

This section reads the structured output from Agent 4 and creates real Gmail drafts using the Gmail API.

**How it works:**
- Reads `gmail_output.txt` (no need to rerun agents)
- Parses each email block using regex
- Calls the Gmail API to create a draft for each one
- Drafts appear in your Gmail inbox ready to review and send

**Note:** Email addresses are formatted using standard `firstname.lastname@company.com` conventions and should be verified before sending.


In [11]:
# Load saved gmail output (no need to rerun agents)
with open("gmail_output.txt", "r") as f:
    gmail_output = f.read()

# Authenticate with Gmail
gmail_service = get_gmail_service()

# Parse email blocks and create drafts
email_blocks = re.split(r'---\s*\n## DRAFT \d+', gmail_output)

drafts_created = 0
skipped = 0
for block in email_blocks:
    try:
        subject_match = re.search(r'\*\*SUBJECT:\*\*\s*(.+)', block)
        body_match = re.search(r'\*\*BODY:\*\*\s*\n(.*?)(?=---|\Z)', block, re.DOTALL)
        email_match = re.search(r'\*\*RECIPIENT EMAIL:\*\*\s*(.+)', block)

        if subject_match and body_match and email_match:
            subject = subject_match.group(1).strip()
            body = body_match.group(1).strip()
            to = email_match.group(1).strip()

            # Skip invalid or placeholder emails
            if '@' not in to or 'to be verified' in to.lower() or '[' in to:
                print(f"⚠️ Skipping invalid email: {to}")
                skipped += 1
                continue

            create_draft(gmail_service, to, subject, body)
            drafts_created += 1
    except Exception as e:
        print(f"❌ Could not create draft: {e}")

print(f"\n🎉 {drafts_created} Gmail drafts created successfully")
print(f"⚠️ {skipped} emails skipped due to invalid or unverified addresses")
print("\nCheck your Gmail Drafts folder to review and send!")

✅ Draft created for sarah.franklin@lattice.com: After the AI product reversal — what's changed internally?
✅ Draft created for sophie.hurcombe@lattice.com: Doing more with a leaner team — how Lattice's ops stack holds up
✅ Draft created for daniel.yanisse@checkr.com: Integrating Truework while running leaner — how's the coordination holding up?
✅ Draft created for luca.bonmassar@checkr.com: Scaling a multi-product platform with a leaner eng team
✅ Draft created for gabe.westmaas@checkr.com: Managing sprint health across multiple product lines post-restructuring
✅ Draft created for ilan.frank@checkr.com: Driving a multi-product vision with a leaner org — what's breaking first?
✅ Draft created for christina.cacioppo@vanta.com: Growing 50% in a year is impressive — keeping everyone on the same page is the hard part
✅ Draft created for iccha.sethi@vanta.com: Engineering at Vanta's growth rate — how are you keeping delivery predictable?
✅ Draft created for jeremy.epling@vanta.com: Prioritiz

---

## 🎉 Pipeline Summary

| Agent | Role | Tools | Output |
|-------|------|-------|--------|
| Agent 1 | Tech Industry Researcher | Tavily Search | Target companies with pain signals |
| Agent 2 | Lead Finder | Tavily Search | Decision-maker contacts |
| Agent 3 | B2B Copywriter | None | Personalized outreach emails |
| Agent 4 | Gmail Drafter | None | Structured email list |

**Key design decisions:**
- Each agent has a single focused responsibility
- Outputs are cached to files — avoiding expensive reruns during development
- Gmail integration uses OAuth with minimal permissions (compose only)
- Email addresses should be verified with Hunter.io or Apollo before sending

**This pipeline is industry-agnostic.** Swap out Agent 1's keywords and Agent 3's product description to target any industry or use case.

---
*Built with CrewAI · Claude Sonnet · Tavily · Gmail API*
